In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, re, json, time
from matplotlib_venn import venn2, venn3
from utility_functions import *

In [2]:
working_folder = "C:/Users/Enrico/OneDrive - UGent/run-ionbot"

combo = []
for dataset_name in ["PXD002057.v0.11.4","PXD005833.v0.11.4","PXD014258.v0.11.4"]:
    openprot_path  = os.path.join(working_folder, dataset_name, f'{dataset_name}-openprot', "combined-results-w-qvalues.csv.gz")
    combo.append(import_pep_IDs(openprot_path, filtering='global', sample_specific_filters=None))
combo = pd.concat(combo)
set(combo.spectrum_file)

{'130327_o2_01_hu_C1_2hr',
 '130327_o2_02_hu_P1_2hr',
 '130327_o2_03_hu_C2_2hr',
 '130327_o2_04_hu_P2_2hr',
 '130327_o2_05_hu_C3_2hr',
 '130327_o2_06_hu_P3_2hr',
 'AM10',
 'AM11',
 'AM12',
 'AM13',
 'AM14',
 'AM15',
 'AM16',
 'AM17',
 'AM18',
 'AM19',
 'AM20',
 'AM21',
 'AM7',
 'AM8',
 'AM9',
 'ESC-HF-Sample-BT474_1',
 'ESC-HF-Sample-BT474_2',
 'ESC-HF-Sample-BT474_3',
 'ESC-HF-Sample-BT474_4',
 'ESC-HF-Sample-BT474_5',
 'ESC-HF-Sample-MCF1',
 'ESC-HF-Sample-MCF2',
 'ESC-HF-Sample-MCF3',
 'ESC-HF-Sample-MCF4',
 'ESC-HF-Sample-MCF5',
 'ESC-HF-SampleHela1',
 'ESC-HF-SampleHela2',
 'ESC-HF-SampleHela3',
 'ESC-HF-SampleHela4',
 'ESC-HF-SampleHela5'}

In [3]:
combo.head()

,ionbot_match_id,spectrum_title,scan,spectrum_file,precursor_mass,charge,database_peptide,matched_peptide,modifications,database,psm_score,proteins,global_q,leadprot,isCanonical,isModified,custom_q,group_qval,modified_peptide,modifications_noRT
0,0_10916_1,130327_o2_01_hu_C1_2hr: controllerType=0 contr...,17820,130327_o2_01_hu_C1_2hr,2377.168920,2,QVQSLTCEVDALKGTNESLER,QVQSLTCEVDALKGTNESLER,7|[4]Carbamidomethyl[C],T,5.46310,P08670;B0YJC5;B0YJC4;Q53HU8;B3KRK8,0.0,P08670,Canonical,Expected,0.0,0.000211,QVQSLTCEVDALKGTNESLER|7|[4]Carbamidomethyl[C],7|[4]Carbamidomethyl[C]
1,0_20604_1,130327_o2_01_hu_C1_2hr: controllerType=0 contr...,30041,130327_o2_01_hu_C1_2hr,2549.169150,2,LCYVALDFEQEMATAASSSSLEK,LCYVALDFEQEMATAASSSSLEK,2|[4]Carbamidomethyl[C],T,5.28342,Q6S8J3;P60709;B3KWQ3;P63261;IP_562814;A0A6Q8PF...,0.0,Q6S8J3,Canonical,Expected,0.0,0.000211,LCYVALDFEQEMATAASSSSLEK|2|[4]Carbamidomethyl[C],2|[4]Carbamidomethyl[C]
2,0_7480_1,130327_o2_01_hu_C1_2hr: controllerType=0 contr...,13373,130327_o2_01_hu_C1_2hr,2743.297338,2,TNHIGHTGYLNTVTVSPDGSLCASGGK,TNHIGHTGYLNTVTVSPDGSLCASGGK,22|[4]Carbamidomethyl[C],T,4.72594,E9KL35;II_604337;II_604338;D6RAC2;J3KPE3;D6RHH...,0.0,E9KL35,Canonical,Expected,0.0,0.000211,TNHIGHTGYLNTVTVSPDGSLCASGGK|22|[4]Carbamidomet...,22|[4]Carbamidomethyl[C]
3,0_19255_1,130327_o2_01_hu_C1_2hr: controllerType=0 contr...,28343,130327_o2_01_hu_C1_2hr,3230.457525,3,CPEALFQPSFLGMESCGIHETTFNSIMK,CPEALFQPSFLGMESCGIHETTFNSIMK,1|[4]Carbamidomethyl[C]||16|[4]Carbamidomethyl[C],T,4.55456,P60709;B3KWQ3;P63261;A0A6Q8PFE4;A0A2R8Y793;E7E...,0.0,P60709,Canonical,Expected,0.0,0.000211,CPEALFQPSFLGMESCGIHETTFNSIMK|1|[4]Carbamidomet...,1|[4]Carbamidomethyl[C]||16|[4]Carbamidomethyl[C]
4,0_19393_1,130327_o2_01_hu_C1_2hr: controllerType=0 contr...,28514,130327_o2_01_hu_C1_2hr,3081.374925,3,VDEFPLCGHMVSDEYEQLSSEALEAAR,VDEFPLCGHMVSDEYEQLSSEALEAAR,7|[4]Carbamidomethyl[C],T,4.44163,P27635;II_558235;II_558245;IP_601897;IP_691616...,0.0,P27635,Canonical,Expected,0.0,0.000211,VDEFPLCGHMVSDEYEQLSSEALEAAR|7|[4]Carbamidometh...,7|[4]Carbamidomethyl[C]


In [4]:
combo.isModified.value_counts()

isModified
Unmodified    214450
Unexpected     83103
Expected       32129
Name: count, dtype: int64

In [5]:
combo.isModified.value_counts(normalize=True)

isModified
Unmodified    0.650475
Unexpected    0.252070
Expected      0.097455
Name: proportion, dtype: float64

In [6]:
combo_noncanon = combo[combo.isCanonical=='NonCanonical'].copy(deep=True)

In [8]:
combo_noncanon.isModified.value_counts(normalize=True)

isModified
Unexpected    0.573465
Expected      0.247873
Unmodified    0.178662
Name: proportion, dtype: float64